In [7]:
import torch
import torch.nn as nn 
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import models
import sys 
from torch.utils.data import DataLoader

sys.path.append("../src")
from model import HybridModel
from data import TrainDataset, TestDataset
from preprocess import split_dataset
from pathlib import Path

In [2]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

# Model 1
model1 = HybridModel()
model1_point = torch.load("../model/model.pth")
model1.load_state_dict(model1_point)
model1.to(device)
model1.eval()

# Model 2
model2 = models.convnext_tiny(pretrained=True)
model2_point = torch.load("../model/ConvNeXt.pth")
model2.load_state_dict(model2_point)
model2.to(device)
model2.eval()

# Model 3
model3 = models.resnet50(pretrained=True)
model3_point = torch.load("../model/ResNet.pth")
model3.load_state_dict(model3_point)
model3.to(device)
model3.eval()


C:\Users\lenov\AppData\Local\Temp\ipykernel_29832\2342995906.py:8: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model1_point = torch.load("../model/model.pth")
c:\Users\len

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
def evaluate(model, loader, device):
    model.eval()

    criterion = nn.CrossEntropyLoss()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():as
        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device, dtype=torch.long)

            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, correct / total

In [ ]:
SEED = 42

train_url = Path("../data/Training").resolve()

full_train_dataset = TrainDataset(root=train_url, transform=None)
train_dataset, val_dataset = split_dataset(full_train_dataset, train_ratio=0.8, seed=SEED)

loader_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, generator=loader_generator)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)



In [14]:
results = {}

models = {
    "HybridModel": model1,
    "ConvNeXt": model2,
    "ResNet50": model3
}

for name, model in models.items():

    train_loss, train_acc = evaluate(
        model, train_loader, device
    )

    val_loss, val_acc = evaluate(
        model, val_loader, device
    )

    results[name] = {
        "train_loss": train_loss,
        "val_loss": val_loss,
        "train_acc": train_acc * 100,
        "val_acc": val_acc * 100
    }

    print(f"\n{name}")
    print(f"Train Loss : {train_loss:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Train Acc  : {train_acc:.2f}")
    print(f"Val Acc    : {val_acc:.2f}")


HybridModel
Train Loss : 0.0048
Val Loss   : 0.0288
Train Acc  : 99.82
Val Acc    : 99.11

ConvNeXt
Train Loss : 0.0052
Val Loss   : 0.0251
Train Acc  : 99.82
Val Acc    : 99.20

ResNet50
Train Loss : 0.0068
Val Loss   : 0.0288
Train Acc  : 99.82
Val Acc    : 99.02
